# Foundation Shade Recommender: Interactive Demo

This notebook follows one clear path from a selfie to a ranked list of
foundation colours. It is the best starting point for readers.

**Method:** MediaPipe landmarks → cheek/forehead pixels → median RGB →
CIELAB → CIEDE2000 product ranking.

The result is a colour-similarity prototype, not a purchasing guarantee.
Lighting, camera processing, makeup, and catalogue colour quality all
affect the result.


## 1. Setup

When running in Colab after publishing this project, replace
`YOUR_USERNAME` below with your GitHub username. Local users should run
the notebook from inside the cloned repository.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/amafteeva/Predictive-beauty-analysis-using-computer-vision-and-machine-learning-.git"


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return None


project_root = find_project_root(Path.cwd())

        
project_root = Path("/content/foundation-shade-recommender")
if not project_root.exists():
        subprocess.run(["git", "clone", REPOSITORY_URL, str(project_root)], check=True)

if project_root is None:
    raise FileNotFoundError("Run this notebook from inside the project folder.")

os.chdir(project_root)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    check=True,
)
print("Project root:", project_root)


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from foundation_matcher.data import load_foundation_catalog
from foundation_matcher.face import (
    create_face_landmarker,
    download_face_landmarker,
    extract_skin_tone,
)
from foundation_matcher.recommender import recommend_foundations
from foundation_matcher.visualization import (
    plot_catalog_lab,
    plot_match_swatches,
    plot_skin_preview,
)


## 2. Prepare the foundation catalogue

The source catalogue stores digital HEX colours. The project cleans those
values and converts them to CIELAB, where numerical distance is more
closely related to perceived colour difference than ordinary RGB distance.


In [ ]:
products = load_foundation_catalog()

print(f"Catalogue rows: {len(products):,}")
print(f"Brands: {products['brand'].nunique():,}")
print(f"Unique HEX colours: {products['hex'].nunique():,}")

display(
    products[
        ["brand", "product", "hex", "lab_L", "lab_a", "lab_b"]
    ].head()
)


In [ ]:
display(products[["lab_L", "lab_a", "lab_b"]].describe().round(2))
plot_catalog_lab(products)
plt.show()


## 3. Select a selfie

For a better estimate, use an unfiltered, front-facing image taken in
even natural light. Avoid dramatic shadows and coloured lighting.

The image remains in the current runtime unless you explicitly save it.
Do not commit personal images to GitHub.


In [ ]:
model_path = download_face_landmarker("models/face_landmarker.task")

if "google.colab" in sys.modules:
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No image was uploaded.")
    image_path = Path(next(iter(uploaded)))
else:
    # Change this path when running locally.
    image_path = Path("data/example_selfie.jpg")
    if not image_path.exists():
        raise FileNotFoundError(
            "Set image_path to a local selfie before running this cell."
        )

print("Selected image:", image_path)


## 4. Detect facial regions and estimate skin colour

The green circles show the sampled regions: both cheeks and the centre
forehead. The estimator trims the darkest and brightest 5% of sampled
pixels and uses the median of the remainder.


In [ ]:
landmarker = create_face_landmarker(model_path)
try:
    skin_tone = extract_skin_tone(image_path, landmarker)
finally:
    landmarker.close()

print("Detected RGB:", skin_tone.rgb.tolist())
print("Detected LAB:", skin_tone.lab.round(2).tolist())
plot_skin_preview(skin_tone)
plt.show()


## 5. Rank foundation colours

CIEDE2000 compares the extracted LAB colour with every catalogue colour.
Smaller Delta E values indicate closer perceptual similarity.


In [ ]:
TOP_N = 5

recommendations = recommend_foundations(
    skin_tone.lab,
    products,
    top_n=TOP_N,
)

display(
    recommendations[
        ["brand", "product", "hex", "color_distance"]
    ].style.format({"color_distance": "{:.2f}"})
)
plot_match_swatches(skin_tone.rgb, recommendations)
plt.show()


## 6. Interpretation

The first row is the closest digital colour in the catalogue, not
necessarily the best real-world product. Formula, oxidation, coverage,
undertone preferences, lighting, price, and availability are outside the
current dataset.

A production version should use calibrated images and professionally
labelled foundation matches rather than relying only on digital HEX values.
